# 4. LangChain Memory - Gestión de Contexto Conversacional

## Objetivos de Aprendizaje
- Comprender la importancia de la memoria en conversaciones con LLMs
- Implementar diferentes tipos de memoria con LangChain
- Gestionar el contexto de conversaciones largas
- Optimizar el uso de tokens con estrategias de memoria

## ¿Por qué es Importante la Memoria?

Los LLMs son **stateless** por naturaleza: no recuerdan conversaciones anteriores. La memoria permite:
- **Contexto conversacional**: Referirse a mensajes anteriores
- **Personalización**: Recordar preferencias del usuario
- **Continuidad**: Mantener hilos de conversación coherentes
- **Experiencia natural**: Conversaciones que se sienten humanas

## Tipos de Memoria en LangChain

1. **ConversationBufferMemory**: Mantiene todo el historial
2. **ConversationSummaryMemory**: Resume conversaciones largas
3. **ConversationBufferWindowMemory**: Mantiene solo los N mensajes más recientes
4. **ConversationSummaryBufferMemory**: Combina resumen + buffer reciente

In [1]:
# Importar bibliotecas necesarias para memoria
import os

# Carga de credenciales: funciona igual en Google Colab y en local (.env)
try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY (Colab: Secrets · local: archivo .env)"

MODELO_RAPIDO = os.getenv("GROQ_MODEL_FAST", "llama-3.1-8b-instant")

from langchain_groq import ChatGroq
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

print("✓ Bibliotecas de memoria importadas correctamente")

✓ Bibliotecas de memoria importadas correctamente


In [2]:
# Configuración del modelo para memoria
# Usamos el modelo rápido: los ejemplos de memoria hacen muchas llamadas cortas
# ChatGroq lee GROQ_API_KEY del entorno automáticamente
try:
    llm = ChatGroq(
        model=MODELO_RAPIDO,
        temperature=0.1
    )

    print("✓ Modelo configurado para experimentos de memoria")
    print(f"Modelo: {llm.model_name}")

except Exception as e:
    print(f"✗ Error en configuración: {e}")
    print("Verifica la variable de entorno GROQ_API_KEY")

✓ Modelo configurado para experimentos de memoria
Modelo: llama-3.1-8b-instant


In [3]:
# Prompt con historial + entrada del usuario
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Cadena = prompt + modelo
chain = prompt | llm

# Almacén de historiales
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Envolver con memoria
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

/Users/giocrisraigodoy/Documents/DUOC/2026-1/INGENIERIA DE SOLUCIONES CON INTELIGENCIA ARTIFICIAL/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3747: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 1. ConversationBufferMemory - Memoria Completa

Esta memoria mantiene **todo** el historial de la conversación. Es la más simple pero puede consumir muchos tokens.

In [4]:
# Ejemplo básico con RunnableWithMessageHistory

# Prompt con historial + entrada
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Cadena = prompt + modelo
chain = prompt | llm

# Almacén de memorias por sesión
store = {}

def get_session_history(session_id: str):
    """Devuelve (o crea) el historial completo para la sesión."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Envolver con RunnableWithMessageHistory
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def ejemplo_buffer_memory():
    print("=== CONVERSATIONBUFFERMEMORY ===")
    print("Mantiene todo el historial de conversación\n")
    
    session_id = "demo_session"

    try:
        # Primera interacción
        print("1. Primera pregunta:")
        response1 = conversation.invoke(
            {"input": "Mi nombre es Ana y soy programadora Python"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response1.content}\n")

        # Segunda interacción
        print("2. Segunda pregunta:")
        response2 = conversation.invoke(
            {"input": "¿Cuál es mi nombre y profesión?"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response2.content}\n")

        # Tercera interacción
        print("3. Tercera pregunta:")
        response3 = conversation.invoke(
            {"input": "¿Qué lenguaje de programación mencioné?"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response3.content}\n")

        # Mostrar historial
        print("=== CONTENIDO DE LA MEMORIA ===")
        history = store[session_id].messages
        for i, msg in enumerate(history, 1):
            print(f"{i}. {msg.type}: {msg.content}")

    except Exception as e:
        print(f"Error: {e}")

# Ejecutar
ejemplo_buffer_memory()


=== CONVERSATIONBUFFERMEMORY ===
Mantiene todo el historial de conversación

1. Primera pregunta:


Respuesta: Hola Ana, soy un asistente útil y estoy aquí para ayudarte con cualquier pregunta o problema que tengas relacionado con la programación en Python. ¿En qué puedo ayudarte hoy? ¿Estás trabajando en un proyecto específico o tienes alguna duda sobre un tema en particular?

2. Segunda pregunta:


Respuesta: Tu nombre es Ana y eres programadora Python.

3. Tercera pregunta:


Respuesta: Mencionaste Python como lenguaje de programación.

=== CONTENIDO DE LA MEMORIA ===
1. human: Mi nombre es Ana y soy programadora Python
2. ai: Hola Ana, soy un asistente útil y estoy aquí para ayudarte con cualquier pregunta o problema que tengas relacionado con la programación en Python. ¿En qué puedo ayudarte hoy? ¿Estás trabajando en un proyecto específico o tienes alguna duda sobre un tema en particular?
3. human: ¿Cuál es mi nombre y profesión?
4. ai: Tu nombre es Ana y eres programadora Python.
5. human: ¿Qué lenguaje de programación mencioné?
6. ai: Mencionaste Python como lenguaje de programación.


## 2. ConversationBufferWindowMemory - Ventana Deslizante

Esta memoria mantiene solo los **N mensajes más recientes**, útil para controlar el uso de tokens.

In [5]:
# Prompt con historial + entrada
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Cadena = prompt + modelo
chain = prompt | llm

# Almacén de memorias por sesión
store = {}

class WindowChatMessageHistory(BaseChatMessageHistory):
    """Historial de chat que mantiene solo los últimos k intercambios."""
    
    def __init__(self, k: int = 2):
        self.k = k
        self._messages = []
    
    @property
    def messages(self):
        # Mantener solo los últimos k intercambios (k*2 mensajes: user + assistant)
        return self._messages[-(self.k * 2):]
    
    def add_message(self, message):
        self._messages.append(message)
    
    def clear(self):
        self._messages.clear()

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """Devuelve el historial de ventana para la sesión."""
    if session_id not in store:
        store[session_id] = WindowChatMessageHistory(k=2)
    return store[session_id]

# Envolver con RunnableWithMessageHistory
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# Ejemplo
def ejemplo_window_memory():
    print("=== CONVERSATION BUFFER WINDOW MEMORY (k=2) ===")
    print("Mantiene solo los 2 intercambios más recientes\n")
    
    session_id = "demo_window"
    inputs = [
        "Mi nombre es Carlos y tengo 30 años",
        "Trabajo como diseñador gráfico", 
        "Me gusta el café y la música jazz",
        "¿Puedes recordar mi edad?",
        "¿Cuál es mi profesión?"
    ]
    
    try:
        for i, user_input in enumerate(inputs, 1):
            print(f"{'='*20} INTERACCIÓN {i} {'='*20}")
            print(f"👤 Usuario: {user_input}")
            
            response = conversation.invoke(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            )
            print(f"🤖 Asistente: {response.content}\n")
            
            # Obtener el historial
            history = get_session_history(session_id)
            
            # Mostrar comparación clara
            total_messages = len(history._messages)
            visible_messages = len(history.messages)
            
            print(f"📊 ESTADO DE LA MEMORIA:")
            print(f"   💾 Total almacenado: {total_messages} mensajes")
            print(f"   👁️  Visible al modelo: {visible_messages} mensajes")
            print(f"   🗑️  Mensajes descartados: {total_messages - visible_messages}")
            
            # Mensajes almacenados totalmente
            print(f"\n📚 HISTORIAL COMPLETO ALMACENADO ({total_messages} mensajes):")
            if total_messages == 0:
                print("     (Ningún mensaje aún)")
            else:
                for j, msg in enumerate(history._messages, 1):
                    role = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                    content = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
                    # Marcar si está en la ventana visible
                    is_visible = j > total_messages - visible_messages
                    marker = "✅" if is_visible else "❌"
                    print(f"     {j}. {marker} {role}: {content}")
            
            # Lo que ve el modelo
            print(f"\n🔍 VENTANA VISIBLE AL MODELO ({visible_messages} mensajes):")
            if visible_messages == 0:
                print("     (Ningún mensaje visible)")
            else:
                for j, msg in enumerate(history.messages, 1):
                    role = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                    content = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
                    print(f"     {j}. ✅ {role}: {content}")
            
            print("\n" + "="*60 + "\n")
            
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar
ejemplo_window_memory()

=== CONVERSATION BUFFER WINDOW MEMORY (k=2) ===
Mantiene solo los 2 intercambios más recientes

==================== INTERACCIÓN 1 ====================
👤 Usuario: Mi nombre es Carlos y tengo 30 años


/Users/giocrisraigodoy/Documents/DUOC/2026-1/INGENIERIA DE SOLUCIONES CON INTELIGENCIA ARTIFICIAL/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3747: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy? ¿Tienes algún tema en particular que te gustaría discutir o necesitas información sobre algo?

📊 ESTADO DE LA MEMORIA:
   💾 Total almacenado: 2 mensajes
   👁️  Visible al modelo: 2 mensajes
   🗑️  Mensajes descartados: 0

📚 HISTORIAL COMPLETO ALMACENADO (2 mensajes):
     1. ✅ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ✅ 🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy...

🔍 VENTANA VISIBLE AL MODELO (2 mensajes):
     1. ✅ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ✅ 🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy...


==================== INTERACCIÓN 2 ====================
👤 Usuario: Trabajo como diseñador gráfico


🤖 Asistente: Interesante, Carlos. Ser diseñador gráfico es un trabajo creativo y emocionante. ¿Te gusta tu trabajo actualmente? ¿Qué tipo de proyectos te gustan trabajar más, como logotipos, carteles, sitios web o algo más?

¿Tienes alguna herramienta o software favorito para realizar tus diseños, como Adobe Creative Cloud, Sketch o Figma?

📊 ESTADO DE LA MEMORIA:
   💾 Total almacenado: 4 mensajes
   👁️  Visible al modelo: 4 mensajes
   🗑️  Mensajes descartados: 0

📚 HISTORIAL COMPLETO ALMACENADO (4 mensajes):
     1. ✅ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ✅ 🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy...
     3. ✅ 👤 Usuario: Trabajo como diseñador gráfico
     4. ✅ 🤖 Asistente: Interesante, Carlos. Ser diseñador gráfico es un trabajo cre...

🔍 VENTANA VISIBLE AL MODELO (4 mensajes):
     1. ✅ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ✅ 🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy...
     3. ✅ 👤 Us

🤖 Asistente: Un gusto compartido, Carlos. El café y la música jazz son una excelente combinación para relajarse y disfrutar del momento. ¿Tienes un café favorito en tu ciudad o prefieres prepararlo en casa? ¿Y qué tipo de música jazz te gusta más, como Miles Davis, John Coltrane o Billie Holiday?

¿Te gusta asistir a conciertos de jazz en vivo o prefieres escuchar música en casa?

📊 ESTADO DE LA MEMORIA:
   💾 Total almacenado: 6 mensajes
   👁️  Visible al modelo: 4 mensajes
   🗑️  Mensajes descartados: 2

📚 HISTORIAL COMPLETO ALMACENADO (6 mensajes):
     1. ❌ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ❌ 🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy...
     3. ✅ 👤 Usuario: Trabajo como diseñador gráfico
     4. ✅ 🤖 Asistente: Interesante, Carlos. Ser diseñador gráfico es un trabajo cre...
     5. ✅ 👤 Usuario: Me gusta el café y la música jazz
     6. ✅ 🤖 Asistente: Un gusto compartido, Carlos. El café y la música jazz son un...

🔍 VENTANA VISIBLE

🤖 Asistente: Lo siento, pero no tengo la capacidad de recordar información personal como tu edad. Nuestro conversatorio comenzó recientemente y no tengo acceso a información previa sobre ti. Si deseas compartir tu edad, estaré encantado de saberlo.

📊 ESTADO DE LA MEMORIA:
   💾 Total almacenado: 8 mensajes
   👁️  Visible al modelo: 4 mensajes
   🗑️  Mensajes descartados: 4

📚 HISTORIAL COMPLETO ALMACENADO (8 mensajes):
     1. ❌ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ❌ 🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy...
     3. ❌ 👤 Usuario: Trabajo como diseñador gráfico
     4. ❌ 🤖 Asistente: Interesante, Carlos. Ser diseñador gráfico es un trabajo cre...
     5. ✅ 👤 Usuario: Me gusta el café y la música jazz
     6. ✅ 🤖 Asistente: Un gusto compartido, Carlos. El café y la música jazz son un...
     7. ✅ 👤 Usuario: ¿Puedes recordar mi edad?
     8. ✅ 🤖 Asistente: Lo siento, pero no tengo la capacidad de recordar informació...

🔍 VENTANA VISIBLE

🤖 Asistente: No tengo información sobre tu profesión. Nuestro conversatorio comenzó recientemente y no tengo acceso a información previa sobre ti. Si deseas hablar sobre tu trabajo o intereses, estaré encantado de escucharte.

📊 ESTADO DE LA MEMORIA:
   💾 Total almacenado: 10 mensajes
   👁️  Visible al modelo: 4 mensajes
   🗑️  Mensajes descartados: 6

📚 HISTORIAL COMPLETO ALMACENADO (10 mensajes):
     1. ❌ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ❌ 🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy...
     3. ❌ 👤 Usuario: Trabajo como diseñador gráfico
     4. ❌ 🤖 Asistente: Interesante, Carlos. Ser diseñador gráfico es un trabajo cre...
     5. ❌ 👤 Usuario: Me gusta el café y la música jazz
     6. ❌ 🤖 Asistente: Un gusto compartido, Carlos. El café y la música jazz son un...
     7. ✅ 👤 Usuario: ¿Puedes recordar mi edad?
     8. ✅ 🤖 Asistente: Lo siento, pero no tengo la capacidad de recordar informació...
     9. ✅ 👤 Usuario: ¿Cuál es mi profes

## 3. ConversationSummaryMemory - Resumen Inteligente

Esta memoria **resume** conversaciones largas en lugar de mantener todo el texto completo, ahorrando tokens significativamente.

In [6]:

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Función para resumir automáticamente cuando hay muchos mensajes
def auto_summarize(session_id: str, max_messages=6):
    history = get_session_history(session_id)
    
    if len(history.messages) > max_messages:
        # Mensajes a resumir (todos excepto los últimos 2)
        messages_to_summarize = history.messages[:-2]
        
        # Crear texto para resumir
        conversation_text = ""
        for msg in messages_to_summarize:
            role = "Usuario" if msg.type == "human" else "Asistente"
            conversation_text += f"{role}: {msg.content}\n"
        
        # Generar resumen
        summary_response = llm.invoke(f"Resume esta conversación en 2-3 líneas:\n{conversation_text}")
        summary = summary_response.content
        
        # Reemplazar mensajes antiguos con el resumen
        recent_messages = history.messages[-2:]
        history.clear()
        history.add_ai_message(f"[RESUMEN]: {summary}")
        history.messages.extend(recent_messages)

# Crear conversación
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

conversation = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def ejemplo_summary_memory():
    print("=== CONVERSATION SUMMARY MEMORY ===")
    print("Resume conversaciones largas para ahorrar tokens\n")
    
    session_id = "summary_session"
    
    # Conversación de ejemplo
    inputs = [
        "Hola, soy María González, ingeniera de software de 35 años",
        "Trabajo en una startup de fintech en Madrid desarrollando pagos digitales",
        "Usamos React, Node.js, Docker y Kubernetes en nuestros proyectos",
        "Mi mayor desafío es la latencia en transacciones internacionales",
        "También trabajo en mejorar la UX de nuestra app móvil",
        "¿Puedes resumir quién soy y cuáles son mis principales desafíos?"
    ]
    
    try:
        for i, user_input in enumerate(inputs, 1):
            print(f"{'='*15} INTERACCIÓN {i} {'='*15}")
            print(f"👤 Usuario: {user_input}")
            
            # Resumir automáticamente si es necesario
            auto_summarize(session_id)
            
            response = conversation.invoke(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            )
            print(f"🤖 Asistente: {response.content}\n")
            
            # Mostrar estado de la memoria
            history = get_session_history(session_id)
            total_messages = len(history.messages)
            
            print(f"📊 ESTADO DE LA MEMORIA:")
            print(f"   💾 Total mensajes: {total_messages}")
            
            # Verificar si hay resumen
            has_summary = any("[RESUMEN]" in msg.content for msg in history.messages if hasattr(msg, 'content'))
            print(f"   📝 Tiene resumen: {'✅ Sí' if has_summary else '❌ No'}")
            
            print(f"\n💬 CONTENIDO ACTUAL DE LA MEMORIA:")
            for j, msg in enumerate(history.messages, 1):
                role = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                content = msg.content
                
                # Destacar si es un resumen
                if "[RESUMEN]" in content:
                    role = "📝 Resumen"
                    content = content.replace("[RESUMEN]: ", "")
                
                # Truncar si es muy largo
                if len(content) > 80:
                    content = content[:80] + "..."
                
                print(f"   {j}. {role}: {content}")
            
            print("\n" + "="*50 + "\n")
            
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar
ejemplo_summary_memory()

=== CONVERSATION SUMMARY MEMORY ===
Resume conversaciones largas para ahorrar tokens

=============== INTERACCIÓN 1 ===============
👤 Usuario: Hola, soy María González, ingeniera de software de 35 años


/Users/giocrisraigodoy/Documents/DUOC/2026-1/INGENIERIA DE SOLUCIONES CON INTELIGENCIA ARTIFICIAL/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3747: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


🤖 Asistente: Hola María, un placer conocerte. ¿En qué puedo ayudarte hoy? ¿Tienes algún proyecto de software en el que necesites asistencia o estás buscando información sobre algún tema específico relacionado con la ingeniería de software?

📊 ESTADO DE LA MEMORIA:
   💾 Total mensajes: 2
   📝 Tiene resumen: ❌ No

💬 CONTENIDO ACTUAL DE LA MEMORIA:
   1. 👤 Usuario: Hola, soy María González, ingeniera de software de 35 años
   2. 🤖 Asistente: Hola María, un placer conocerte. ¿En qué puedo ayudarte hoy? ¿Tienes algún proye...


=============== INTERACCIÓN 2 ===============
👤 Usuario: Trabajo en una startup de fintech en Madrid desarrollando pagos digitales


🤖 Asistente: ¡Eso suena emocionante! La fintech es un sector en constante evolución y los pagos digitales están cambiando la forma en que las personas realizan transacciones financieras.

¿Cuál es tu rol específico en la startup? ¿Eres desarrolladora de backend, frontend o estás trabajando en la integración de pagos con diferentes plataformas?

¿Qué desafíos estás enfrentando actualmente en tu trabajo y cómo puedo ayudarte?

📊 ESTADO DE LA MEMORIA:
   💾 Total mensajes: 4
   📝 Tiene resumen: ❌ No

💬 CONTENIDO ACTUAL DE LA MEMORIA:
   1. 👤 Usuario: Hola, soy María González, ingeniera de software de 35 años
   2. 🤖 Asistente: Hola María, un placer conocerte. ¿En qué puedo ayudarte hoy? ¿Tienes algún proye...
   3. 👤 Usuario: Trabajo en una startup de fintech en Madrid desarrollando pagos digitales
   4. 🤖 Asistente: ¡Eso suena emocionante! La fintech es un sector en constante evolución y los pag...


=============== INTERACCIÓN 3 ===============
👤 Usuario: Usamos React, Node.js, Docker y 

🤖 Asistente: Un equipo tecnológico muy moderno y escalable. React es una excelente opción para la interfaz de usuario, Node.js es un servidor de alto rendimiento y Docker y Kubernetes te permiten automatizar la implementación y la gestión de tus aplicaciones en la nube.

¿Cómo estás utilizando Docker y Kubernetes en tu proyecto? ¿Has implementado algún patrón de arquitectura específico, como microservicios o serverless?

¿Te gustaría hablar sobre algún tema en particular, como seguridad en la nube, rendimiento de la aplicación o integración con APIs de terceros?

📊 ESTADO DE LA MEMORIA:
   💾 Total mensajes: 6
   📝 Tiene resumen: ❌ No

💬 CONTENIDO ACTUAL DE LA MEMORIA:
   1. 👤 Usuario: Hola, soy María González, ingeniera de software de 35 años
   2. 🤖 Asistente: Hola María, un placer conocerte. ¿En qué puedo ayudarte hoy? ¿Tienes algún proye...
   3. 👤 Usuario: Trabajo en una startup de fintech en Madrid desarrollando pagos digitales
   4. 🤖 Asistente: ¡Eso suena emocionante! La fintech

🤖 Asistente: La latencia en transacciones internacionales es un desafío común en la industria de los pagos digitales. La latencia se debe a la necesidad de realizar conversaciones con bancos y sistemas de pago en diferentes países, lo que puede llevar a retrasos en la procesamiento de transacciones.

¿Has considerado utilizar tecnologías como la programación en tiempo real (real-time programming) o la programación asincrónica para mejorar la velocidad de procesamiento de transacciones?

También podrías considerar utilizar servicios de pago que ofrecen APIs de alta velocidad y baja latencia, como Stripe o PayPal. ¿Has investigado sobre estas opciones o tienes alguna otra estrategia en mente para reducir la latencia en transacciones internacionales?

¿Te gustaría hablar sobre cómo podrías implementar una solución de baja latencia en tu proyecto utilizando tecnologías como Node.js, Docker y Kubernetes?

📊 ESTADO DE LA MEMORIA:
   💾 Total mensajes: 8
   📝 Tiene resumen: ❌ No

💬 CONTENIDO A

🤖 Asistente: Me alegra saber que estás trabajando en mejorar la experiencia del usuario (UX) de tu app móvil. La UX es fundamental para retener a los usuarios y aumentar la satisfacción general.

¿Cuáles son algunos de los problemas o áreas de mejora que has identificado en la UX de tu app móvil? ¿Estás trabajando en mejorar la navegación, la interfaz de usuario, la velocidad de carga o algo más?

También podrías considerar utilizar herramientas de análisis de usuario, como Google Analytics o Hotjar, para entender mejor los comportamientos y preferencias de tus usuarios. ¿Has utilizado alguna de estas herramientas o tienes alguna otra estrategia en mente para mejorar la UX de tu app móvil?

¿Te gustaría hablar sobre cómo podrías utilizar principios de diseño de usuario (UX design) y patrones de diseño para mejorar la experiencia del usuario en tu app móvil?

📊 ESTADO DE LA MEMORIA:
   💾 Total mensajes: 5
   📝 Tiene resumen: ✅ Sí

💬 CONTENIDO ACTUAL DE LA MEMORIA:
   1. 📝 Resumen: Aquí 

🤖 Asistente: Claro, te resumiré quién eres y cuáles son tus principales desafíos:

**Quién eres:** Eres María González, una ingeniera de software de 35 años que trabaja en una startup de fintech en Madrid. Desarrollas pagos digitales utilizando tecnologías como React, Node.js, Docker y Kubernetes.

**Principales desafíos:**

1. **Latencia en transacciones internacionales**: Buscas mejorar la velocidad de procesamiento de transacciones internacionales para reducir la latencia y mejorar la experiencia del usuario.
2. **Mejora de la UX de la app móvil**: Trabajas en mejorar la experiencia del usuario en la app móvil, identificando áreas de mejora en la navegación, la interfaz de usuario y la velocidad de carga.

Espero que esta información sea útil. ¿Hay algo más en lo que pueda ayudarte?

📊 ESTADO DE LA MEMORIA:
   💾 Total mensajes: 7
   📝 Tiene resumen: ✅ Sí

💬 CONTENIDO ACTUAL DE LA MEMORIA:
   1. 📝 Resumen: Aquí te presento un resumen de la conversación en 3 líneas:

María González, i

## Consideraciones Técnicas y Mejores Prácticas

### Selección del Tipo de Memoria

| Tipo | Cuándo Usarlo | Ventajas | Desventajas |
|------|---------------|----------|-------------|
| **Buffer** | Conversaciones cortas | Contexto completo | Alto consumo de tokens |
| **Window** | Contexto reciente importante | Eficiente en tokens | Puede perder información clave |
| **Summary** | Conversaciones muy largas | Balance eficiencia/contexto | Pérdida de detalles específicos |

### Mejores Prácticas:

1. **Gestión de Tokens**:
   - Monitorea el uso de tokens regularmente
   - Establece límites máximos para evitar costos excesivos
   - Considera el costo vs. calidad del contexto

2. **Selección Estratégica**:
   - Usa Buffer para sesiones cortas e importantes
   - Usa Window para conversaciones con contexto limitado
   - Usa Summary para sesiones largas de asistencia

3. **Optimización**:
   - Limpia memoria periódicamente si es necesario
   - Implementa estrategias híbridas según el caso de uso
   - Considera almacenamiento persistente para memoria a largo plazo

## Ejercicios Prácticos

### Ejercicio 1: Análisis de Consumo
Implementa un sistema que monitoree y reporte el uso de tokens con diferentes tipos de memoria.

### Ejercicio 2: Memoria Híbrida
Diseña una estrategia que combine multiple tipos de memoria según el contexto.

### Ejercicio 3: Persistencia
Extiende el chatbot para guardar y cargar memoria entre sesiones.

## Conceptos Clave Aprendidos

1. **Importancia de la memoria** en conversaciones naturales
2. **Tipos de memoria** y sus casos de uso específicos
3. **Balance** entre contexto y eficiencia de tokens
4. **Implementación práctica** con LangChain
5. **Estrategias de optimización** para diferentes escenarios

## Conclusión del Módulo IL1.1

Has completado la introducción a LLMs y conexiones API. Los conceptos aprendidos:

1. **APIs directas** vs **frameworks** como LangChain
2. **Streaming** para mejor experiencia de usuario
3. **Memoria** para conversaciones contextuales
4. **Mejores prácticas** de seguridad y optimización

### Próximos Pasos
En **IL1.2** exploraremos técnicas avanzadas de **prompt engineering** incluyendo zero-shot, few-shot, y chain-of-thought prompting.